In [2]:
import sqlite3
import pandas as pd

# Tạo database in-memory
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

# Tạo bảng
cursor.execute("""
CREATE TABLE student (
    student_id INTEGER,
    name TEXT,
    class TEXT,
    course_id INTEGER,
    score REAL
)
""")

cursor.execute("""
CREATE TABLE course (
    id INTEGER,
    course_name TEXT
)
""")

# Insert dữ liệu student
students = [
    (1, "Nguyen Minh Hoang", "May Tinh", 12, 6.7),
    (2, "Tran Thi Lan", "Kinh Te", 34, 9.2),
    (3, "Pham Van Nam", "Toan Tin", None, 7.9),
    (4, "Le Thanh Huyen", "Toan Tin", 20, 7.2),
    (5, "Vu Quoc Anh", "May Tinh", 24, 8.0),
    (6, "Dang Thuy Linh", "May Tinh", 24, 5.5),
    (7, "Bui Tien Dung", "Kinh Te", 34, 9.2),
    (8, "Ho Ngoc Mai", "Toan Tin", 20, 8.8),
    (9, "Duong Huu Phuc", "Kinh Te", None, 7.2),
    (10, "Cao Thi Hanh", "May Tinh", None, 7.0)
]

cursor.executemany("INSERT INTO student VALUES (?, ?, ?, ?, ?)", students)

# Insert dữ liệu course
courses = [
    (12, "Giai tich"),
    (34, "Thong ke"),
    (26, "Tin hoc")
]

cursor.executemany("INSERT INTO course VALUES (?, ?)", courses)

conn.commit()

In [3]:
print("INNER JOIN")
df_inner = pd.read_sql_query("""
SELECT * FROM student s
INNER JOIN course c ON s.course_id = c.id
""", conn)
print(df_inner)

print("\nLEFT JOIN")
df_left = pd.read_sql_query("""
SELECT * FROM student s
LEFT JOIN course c ON s.course_id = c.id
""", conn)
print(df_left)

print("\nRIGHT JOIN (giả lập)")
df_right = pd.read_sql_query("""
SELECT * FROM course c
LEFT JOIN student s ON s.course_id = c.id
""", conn)
print(df_right)

print("\nFULL OUTER JOIN (giả lập)")
df_full = pd.read_sql_query("""
SELECT * FROM student s
LEFT JOIN course c ON s.course_id = c.id
UNION
SELECT * FROM course c
LEFT JOIN student s ON s.course_id = c.id
""", conn)
print(df_full)

INNER JOIN
   student_id               name     class  course_id  score  id course_name
0           1  Nguyen Minh Hoang  May Tinh         12    6.7  12   Giai tich
1           2       Tran Thi Lan   Kinh Te         34    9.2  34    Thong ke
2           7      Bui Tien Dung   Kinh Te         34    9.2  34    Thong ke

LEFT JOIN
   student_id               name     class  course_id  score    id course_name
0           1  Nguyen Minh Hoang  May Tinh       12.0    6.7  12.0   Giai tich
1           2       Tran Thi Lan   Kinh Te       34.0    9.2  34.0    Thong ke
2           3       Pham Van Nam  Toan Tin        NaN    7.9   NaN        None
3           4     Le Thanh Huyen  Toan Tin       20.0    7.2   NaN        None
4           5        Vu Quoc Anh  May Tinh       24.0    8.0   NaN        None
5           6     Dang Thuy Linh  May Tinh       24.0    5.5   NaN        None
6           7      Bui Tien Dung   Kinh Te       34.0    9.2  34.0    Thong ke
7           8        Ho Ngoc Mai  Toan

In [4]:
cursor.execute("""
UPDATE student
SET course_id = 26
WHERE course_id IS NULL
""")
conn.commit()

In [5]:
df_avg_class = pd.read_sql_query("""
SELECT class, AVG(score) AS avg_score
FROM student
GROUP BY class
""", conn)
print(df_avg_class)

      class  avg_score
0   Kinh Te   8.533333
1  May Tinh   6.800000
2  Toan Tin   7.966667


In [6]:
df_course = pd.read_sql_query("""
SELECT c.course_name,
       COUNT(s.student_id) AS total_students,
       AVG(s.score) AS avg_score
FROM student s
JOIN course c ON s.course_id = c.id
GROUP BY c.course_name
""", conn)
print(df_course)

  course_name  total_students  avg_score
0   Giai tich               1   6.700000
1    Thong ke               2   9.200000
2     Tin hoc               3   7.366667


In [7]:
df_rank = pd.read_sql_query("""
SELECT name, score,
CASE
    WHEN score >= 9 THEN 'Xuat sac'
    WHEN score >= 6 THEN 'Tot'
    ELSE 'Kem'
END AS rank
FROM student
""", conn)
print(df_rank)

                name  score      rank
0  Nguyen Minh Hoang    6.7       Tot
1       Tran Thi Lan    9.2  Xuat sac
2       Pham Van Nam    7.9       Tot
3     Le Thanh Huyen    7.2       Tot
4        Vu Quoc Anh    8.0       Tot
5     Dang Thuy Linh    5.5       Kem
6      Bui Tien Dung    9.2  Xuat sac
7        Ho Ngoc Mai    8.8       Tot
8     Duong Huu Phuc    7.2       Tot
9       Cao Thi Hanh    7.0       Tot


In [8]:
df_rank_all = pd.read_sql_query("""
SELECT name, score,
RANK() OVER (ORDER BY score DESC) AS rank
FROM student
""", conn)
print(df_rank_all)

                name  score  rank
0       Tran Thi Lan    9.2     1
1      Bui Tien Dung    9.2     1
2        Ho Ngoc Mai    8.8     3
3        Vu Quoc Anh    8.0     4
4       Pham Van Nam    7.9     5
5     Le Thanh Huyen    7.2     6
6     Duong Huu Phuc    7.2     6
7       Cao Thi Hanh    7.0     8
8  Nguyen Minh Hoang    6.7     9
9     Dang Thuy Linh    5.5    10


In [9]:
df_rank_class = pd.read_sql_query("""
SELECT name, class, score,
RANK() OVER (PARTITION BY class ORDER BY score DESC) AS rank
FROM student
""", conn)
print(df_rank_class)

                name     class  score  rank
0       Tran Thi Lan   Kinh Te    9.2     1
1      Bui Tien Dung   Kinh Te    9.2     1
2     Duong Huu Phuc   Kinh Te    7.2     3
3        Vu Quoc Anh  May Tinh    8.0     1
4       Cao Thi Hanh  May Tinh    7.0     2
5  Nguyen Minh Hoang  May Tinh    6.7     3
6     Dang Thuy Linh  May Tinh    5.5     4
7        Ho Ngoc Mai  Toan Tin    8.8     1
8       Pham Van Nam  Toan Tin    7.9     2
9     Le Thanh Huyen  Toan Tin    7.2     3


In [10]:
df_rank_course = pd.read_sql_query("""
SELECT s.name, c.course_name, s.score,
RANK() OVER (PARTITION BY c.course_name ORDER BY s.score DESC) AS rank
FROM student s
JOIN course c ON s.course_id = c.id
""", conn)
print(df_rank_course)

                name course_name  score  rank
0  Nguyen Minh Hoang   Giai tich    6.7     1
1       Tran Thi Lan    Thong ke    9.2     1
2      Bui Tien Dung    Thong ke    9.2     1
3       Pham Van Nam     Tin hoc    7.9     1
4     Duong Huu Phuc     Tin hoc    7.2     2
5       Cao Thi Hanh     Tin hoc    7.0     3


In [11]:
print("Top 3 cao nhất")
print(pd.read_sql_query("""
SELECT * FROM student
ORDER BY score DESC
LIMIT 3
""", conn))

print("\nTop 3 thấp nhất")
print(pd.read_sql_query("""
SELECT * FROM student
ORDER BY score ASC
LIMIT 3
""", conn))

Top 3 cao nhất
   student_id           name     class  course_id  score
0           2   Tran Thi Lan   Kinh Te         34    9.2
1           7  Bui Tien Dung   Kinh Te         34    9.2
2           8    Ho Ngoc Mai  Toan Tin         20    8.8

Top 3 thấp nhất
   student_id               name     class  course_id  score
0           6     Dang Thuy Linh  May Tinh         24    5.5
1           1  Nguyen Minh Hoang  May Tinh         12    6.7
2          10       Cao Thi Hanh  May Tinh         26    7.0


In [12]:
cursor.execute("ALTER TABLE student ADD COLUMN graduation_date TEXT")

cursor.execute("""
UPDATE student
SET graduation_date = datetime('now',
    CASE
        WHEN score >= 9 THEN '+1 year'
        WHEN score >= 6 THEN '+2 year'
        ELSE '+3 year'
    END
)
""")

conn.commit()

print(pd.read_sql_query("SELECT * FROM student", conn))

   student_id               name     class  course_id  score  \
0           1  Nguyen Minh Hoang  May Tinh         12    6.7   
1           2       Tran Thi Lan   Kinh Te         34    9.2   
2           3       Pham Van Nam  Toan Tin         26    7.9   
3           4     Le Thanh Huyen  Toan Tin         20    7.2   
4           5        Vu Quoc Anh  May Tinh         24    8.0   
5           6     Dang Thuy Linh  May Tinh         24    5.5   
6           7      Bui Tien Dung   Kinh Te         34    9.2   
7           8        Ho Ngoc Mai  Toan Tin         20    8.8   
8           9     Duong Huu Phuc   Kinh Te         26    7.2   
9          10       Cao Thi Hanh  May Tinh         26    7.0   

       graduation_date  
0  2028-04-03 00:20:06  
1  2027-04-03 00:20:06  
2  2028-04-03 00:20:06  
3  2028-04-03 00:20:06  
4  2028-04-03 00:20:06  
5  2029-04-03 00:20:06  
6  2027-04-03 00:20:06  
7  2028-04-03 00:20:06  
8  2028-04-03 00:20:06  
9  2028-04-03 00:20:06  
